# RAS 2026 — Neural Destroy + OR-Tools Repair (NDOR)

**Pipeline:**
1. Greedy → BC warm-start (RL routing policy)
2. **Neural Destroy**: RL SelectionPolicy learns which K demands to remove
3. **OR-Tools Repair**: CP-SAT finds optimal re-assignment of K demands (C++ speed)
4. **SA acceptance**: exp(-Δ/T) annealing for exploration
5. REINFORCE update on SelectionPolicy

**C++ evaluator**: O(1) delta evaluation via maintained block state → 100× faster SA

**Paper**: *Neural Destroy with Optimal Repair for Railroad Blocking Plan Optimization*

## 0. Setup

In [ ]:
# Google Drive 마운트 + repo clone
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_CACHE = '/content/drive/MyDrive/ras2026_cache'
os.makedirs(DRIVE_CACHE, exist_ok=True)
print(f'Drive cache: {DRIVE_CACHE}')

if not os.path.exists('ras2026'):
    !git clone https://github.com/nepersoned/ras2026.git
os.chdir('ras2026')
print('cwd:', os.getcwd())

In [ ]:
!pip install -q pybind11 ortools scipy numpy pandas torch

In [ ]:
# Kaggle credentials (데이터 다운로드 불필요 — Drive에서 직접 마운트)
# 아래 셀에서 Drive 심볼릭 링크로 연결함

In [ ]:
# Drive에서 데이터 심볼릭 링크
import os
DRIVE_DATA = '/content/drive/MyDrive/ras_release_v1.1'
if not os.path.exists('ras_release_v1.1'):
    os.symlink(DRIVE_DATA, 'ras_release_v1.1')
print('Data ready:', os.path.exists('ras_release_v1.1'))

## 1. Build C++ Evaluator

In [ ]:
import subprocess, sys

pybind_inc = subprocess.check_output(
    [sys.executable, '-m', 'pybind11', '--includes']
).decode().strip()

ext_suffix = subprocess.check_output(
    [sys.executable, '-c', 'import sysconfig; print(sysconfig.get_config_var("EXT_SUFFIX"))']
).decode().strip()

out = f'src/evaluator{ext_suffix}'
cmd = f'g++ -O3 -shared -fPIC {pybind_inc} src/evaluator.cpp -o {out}'
print('Building:', cmd)
ret = os.system(cmd)
print('Done!' if ret == 0 else f'ERROR: {ret}')

In [ ]:
import sys
sys.path.insert(0, 'src')
import evaluator as ev
print('C++ evaluator loaded:', ev.__doc__)

## 2. Environment + Precompute

In [ ]:
import sys
sys.path.insert(0, '.')

import math, random, time, json, csv
from pathlib import Path
from copy import deepcopy
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

from solver import (
    load_layer, load_od_matrix, Network, Demand, Solution,
    greedy_construct, evaluate, build_json,
    COMMODITY_TO_BLOCK_TYPE, DIRECT_ONLY,
)
from src.repair import build_candidates, ortools_repair

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
COMMODITY_IDX    = {'Manifest': 0, 'Bulk': 1, 'Intermodal': 2, 'Multilevel': 3}
DIRECT_ONLY_COMM = {'Intermodal', 'Automobile'}

MAX_HUBS = 50
FEAT_DIM = 7 + MAX_HUBS * 4   # 207
SEL_DIM  = FEAT_DIM + 3       # 210


def load_env(layer, multiplier):
    import pickle
    cache_path = f'{DRIVE_CACHE}/od_matrix_{layer}.pkl'

    nodes_df, links_df, demands_scaled, demands_raw, settings = load_layer(layer, multiplier)
    yard_rows = nodes_df[nodes_df['node_type'] == 'yard']
    yard_info = {
        int(r['node_id']): {
            'num_tracks':        float(r.get('num_tracks', 9999) or 9999),
            'handling_capacity': float(r.get('handling_capacity', 1e9) or 1e9),
            'handling_cost':     float(r.get('handling_cost', 0) or 0),
        } for _, r in yard_rows.iterrows()
    }
    origin_ids   = set(demands_scaled['origin_yard_id'].astype(int))
    dest_ids     = set(demands_scaled['dest_yard_id'].astype(int))
    all_yard_ids = sorted(origin_ids | dest_ids)

    # ── Dijkstra with Drive cache ──────────────────────────────────────────────
    if os.path.exists(cache_path):
        print(f'  Loading od_matrix from Drive cache: {cache_path}')
        with open(cache_path, 'rb') as f:
            od_matrix = pickle.load(f)
    else:
        print(f'  Computing od_matrix (will save to Drive cache)...')
        all_od_pairs = {(o,d) for o in all_yard_ids for d in all_yard_ids if o!=d}
        od_matrix    = load_od_matrix(all_od_pairs)
        with open(cache_path, 'wb') as f:
            pickle.dump(od_matrix, f)
        print(f'  Saved to {cache_path}')

    net = Network(nodes_df, links_df, origin_ids, dest_ids, settings, verbose=True)
    demands = [
        Demand(
            idx=idx, demand_id=int(r['demand_id']),
            commodity_type=str(r['block_type']),
            block_type=COMMODITY_TO_BLOCK_TYPE.get(str(r['block_type']), 'Manifest'),
            origin=int(r['origin_yard_id']), dest=int(r['dest_yard_id']),
            volume=int(r['volume']),
            sp_dist=od_matrix.get(
                (int(r['origin_yard_id']), int(r['dest_yard_id'])),
                net.dist(int(r['origin_yard_id']), int(r['dest_yard_id']))
            ),
        ) for idx, (_, r) in enumerate(demands_scaled.iterrows())
    ]
    return dict(net=net, od_matrix=od_matrix, settings=settings,
                yard_info=yard_info, demands=demands, all_yards=all_yard_ids,
                nodes_df=nodes_df, links_df=links_df, demands_raw=demands_raw)


def ranked_hubs(dem, env):
    if dem.commodity_type in DIRECT_ONLY_COMM:
        return []
    scored = []
    for hub in env['all_yards']:
        if hub in (dem.origin, dem.dest): continue
        d1 = env['net'].dist(dem.origin, hub)
        d2 = env['net'].dist(hub, dem.dest)
        if math.isinf(d1) or math.isinf(d2): continue
        scored.append((d1+d2, hub))
    scored.sort()
    return [h for _,h in scored]


def demand_feat(dem, hubs, env):
    od_d  = env['net'].dist(dem.origin, dem.dest)
    od_ic = env['net'].interchanges(dem.origin, dem.dest)
    bt_oh = [1.0 if COMMODITY_TO_BLOCK_TYPE.get(dem.commodity_type,'Manifest')==t else 0.0
             for t in ['Manifest','Bulk','Intermodal','Multilevel']]
    base = [math.log1p(dem.volume),
            math.log1p(od_d if math.isfinite(od_d) else 1e6),
            od_ic/5.0] + bt_oh
    hub_f = [0.0]*(MAX_HUBS*4)
    for j, hub in enumerate(hubs[:MAX_HUBS]):
        d1 = env['net'].dist(dem.origin, hub)
        d2 = env['net'].dist(hub, dem.dest)
        if math.isinf(d1) or math.isinf(d2): continue
        ic1 = env['net'].interchanges(dem.origin, hub)
        ic2 = env['net'].interchanges(hub, dem.dest)
        hc  = env['yard_info'].get(hub,{}).get('handling_cost',0.0)
        hub_f[j*4]  = math.log1p(d1)
        hub_f[j*4+1]= math.log1p(d2)
        hub_f[j*4+2]= (ic1+ic2)/5.0
        hub_f[j*4+3]= hc/500.0
    return base + hub_f


def precompute(env):
    data = []
    for dem in env['demands']:
        if dem.volume <= 0:
            data.append(None); continue
        hubs  = ranked_hubs(dem, env)
        feat  = demand_feat(dem, hubs, env)
        cands = _build_cands(dem, hubs, env)
        data.append({'feat': feat, 'hubs': hubs, 'candidates': cands,
                     'direct_only': dem.commodity_type in DIRECT_ONLY_COMM})
    return data


def _build_cands(dem, hubs, env):
    s  = env['settings']
    tc = float(s.get('transport_cost_coefficient', 1.0))
    ic = float(s.get('interchange_cost', 100))
    M  = float(s.get('stress_penalty_M', 5))
    net= env['net']
    cands = []
    cands.append((None, M * dem.volume * dem.sp_dist))
    od_d = net.dist(dem.origin, dem.dest)
    if not math.isinf(od_d):
        od_ic = net.interchanges(dem.origin, dem.dest)
        cands.append(([(dem.origin, dem.dest)],
                      tc*dem.volume*od_d + ic*dem.volume*od_ic))
    if dem.commodity_type not in DIRECT_ONLY_COMM:
        for hub in hubs[:MAX_HUBS]:
            d1 = net.dist(dem.origin, hub)
            d2 = net.dist(hub, dem.dest)
            if math.isinf(d1) or math.isinf(d2): continue
            ic1= net.interchanges(dem.origin, hub)
            ic2= net.interchanges(hub, dem.dest)
            hc = env['yard_info'].get(hub,{}).get('handling_cost',0.0)
            cands.append(([(dem.origin,hub),(hub,dem.dest)],
                          tc*dem.volume*(d1+d2) + ic*dem.volume*(ic1+ic2) + hc*dem.volume))
    return cands


print('Environment functions ready.')
print(f'Drive cache dir: {DRIVE_CACHE}')

## 3. C++ Solver Setup

In [ ]:
def build_cpp_solver(env, dd, init_routings):
    """
    Package environment data for the C++ RasSolver.
    Returns initialized RasSolver ready for sa_run().
    """
    demands = env['demands']
    s = env['settings']

    # Build demand list
    dem_list = []
    for i, dem in enumerate(demands):
        comm_idx = COMMODITY_IDX.get(COMMODITY_TO_BLOCK_TYPE.get(dem.commodity_type,'Manifest'), 0)
        dem_list.append((
            i, dem.origin, dem.dest, dem.volume,
            dem.sp_dist, comm_idx,
            dem.commodity_type in DIRECT_ONLY_COMM
        ))

    # Build candidate routes per demand
    cand_list = []
    for i, d in enumerate(dd):
        if d is None:
            cand_list.append([(
                True, False, -1, 0.0, 0.0, -1, -1, -1, -1
            )])
            continue
        dem   = demands[i]
        cands = d['candidates']
        cpp_cands = []
        for route, cost in cands:
            if route is None:
                cpp_cands.append((True, False, -1, cost, 0.0, -1, -1, -1, -1))
            elif len(route) == 1:
                o, dst = route[0]
                cpp_cands.append((False, True, -1, cost, 0.0, o, dst, -1, -1))
            else:
                (o1,h),(h2,d2) = route
                hc = env['yard_info'].get(h,{}).get('handling_cost',0.0) * dem.volume
                cpp_cands.append((False, False, h, cost-hc, hc, o1, h, h, d2))
        cand_list.append(cpp_cands)

    # Initial route indices (index into cand_list)
    init_route_idx = []
    for i, routing in enumerate(init_routings):
        d = dd[i]
        if d is None:
            init_route_idx.append(0)
            continue
        # Find matching candidate
        found = 0
        for ri, (route, _) in enumerate(d['candidates']):
            if route == routing:
                found = ri; break
        init_route_idx.append(found)

    # Yard info
    yd_tracks = {k: int(v['num_tracks']) for k,v in env['yard_info'].items()}
    yd_hcap   = {k: float(v['handling_capacity']) for k,v in env['yard_info'].items()}

    # Segment distances from od_matrix
    seg_dist = {f'{o}_{d}': float(dist)
                for (o,d), dist in env['od_matrix'].items()
                if math.isfinite(dist)}

    # Settings
    cpp_settings = ev.Settings()
    cpp_settings.block_fixed_cost   = float(s.get('block_fixed_cost', 1500))
    cpp_settings.unserved_penalty   = float(s.get('stress_penalty_M', 5))
    cpp_settings.min_block_vol_short  = 5.0
    cpp_settings.min_block_vol_medium = 10.0
    cpp_settings.min_block_vol_long   = 15.0

    solver = ev.RasSolver()
    solver.init(dem_list, cand_list, init_route_idx,
                yd_tracks, yd_hcap, seg_dist, cpp_settings)

    print(f'C++ solver init: score={solver.get_score():,.0f}')
    return solver, cand_list


def route_idx_to_routing(route_indices, dd, env):
    """Convert C++ route indices back to Python route format."""
    routings = []
    for i, ri in enumerate(route_indices):
        d = dd[i]
        if d is None:
            routings.append(None); continue
        route, _ = d['candidates'][ri]
        routings.append(route)
    return routings


print('C++ solver setup ready.')

## 4. Neural Destroy Policy + BC Warm-start

In [ ]:
class SelectionPolicy(nn.Module):
    """Learns which demands to destroy (remove from current solution).
    Input: per-demand features + current route state (210-dim)
    Output: importance score per demand → softmax → sampling
    """
    def __init__(self, feat_dim=SEL_DIM, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feat_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),   nn.ReLU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)   # (N,) scores


class RoutingPolicy(nn.Module):
    """BC-pretrained: demand features → which route."""
    def __init__(self, feat_dim=FEAT_DIM, n_actions=None, hidden=256):
        super().__init__()
        # n_actions determined dynamically per demand via candidates
        self.net = nn.Sequential(
            nn.Linear(feat_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),   nn.ReLU(),
            nn.Linear(hidden, MAX_HUBS + 2),
        )
    def forward(self, x):
        return self.net(x)


def make_policies():
    return SelectionPolicy().to(DEVICE), RoutingPolicy().to(DEVICE)


def get_sel_feats(dd, route_indices):
    """Build selection features: static feat + current route state."""
    feats, valid = [], []
    for i, d in enumerate(dd):
        if d is None: continue
        ri    = route_indices[i]
        route, _ = d['candidates'][ri]
        is_unserved = 1.0 if route is None else 0.0
        is_direct   = 1.0 if (route is not None and len(route)==1) else 0.0
        is_hub      = 1.0 if (route is not None and len(route)==2) else 0.0
        feats.append(d['feat'] + [is_direct, is_hub, is_unserved])
        valid.append(i)
    return feats, valid


def pretrain_bc(rout_policy, env, dd, n_epochs=200, lr=1e-3, print_every=50):
    init_sol = greedy_construct(env['demands'], env['net'], env['od_matrix'],
                                env['settings'], env['yard_info'])
    score0, _ = evaluate(init_sol, env['net'], env['od_matrix'],
                         env['settings'], env['yard_info'])

    feats, labels = [], []
    for i, dem in enumerate(env['demands']):
        d = dd[i]
        if d is None: continue
        route  = init_sol.routings[i]
        # Find best matching candidate index
        best_ri = 0
        for ri, (cand_route, _) in enumerate(d['candidates']):
            if cand_route == route:
                best_ri = ri; break
        # Map to the fixed N_ACTIONS policy output
        if route is None:   label = MAX_HUBS + 1
        elif len(route)==1: label = 0
        else:
            hub = route[0][1]
            label = (d['hubs'].index(hub)+1) if hub in d['hubs'][:MAX_HUBS] else MAX_HUBS+1
        feats.append(d['feat'])
        labels.append(min(label, MAX_HUBS+1))

    feat_t  = torch.tensor(feats,  dtype=torch.float32, device=DEVICE)
    label_t = torch.tensor(labels, dtype=torch.long,    device=DEVICE)
    opt = torch.optim.Adam(rout_policy.parameters(), lr=lr)
    print(f'\n── BC ──  N={len(labels)}  greedy={score0:,.0f}')

    for ep in range(1, n_epochs+1):
        rout_policy.train()
        logits = rout_policy(feat_t)
        loss   = F.cross_entropy(logits, label_t)
        opt.zero_grad(); loss.backward(); opt.step()
        if ep % print_every == 0:
            acc = (logits.argmax(1)==label_t).float().mean().item()*100
            print(f'  ep {ep:4d} | loss={loss.item():.4f} | acc={acc:.1f}%')

    print('BC done.')
    return init_sol, score0


print(f'Policies ready.')
print(f'  Selection: {SEL_DIM}→128→128→1')
print(f'  Routing:   {FEAT_DIM}→256→256→{MAX_HUBS+2}')

## 5. NDOR Training Loop

In [ ]:
def ndor_train(sel_policy, env, dd, init_sol, init_score,
               n_episodes=300, sa_iters_per_ep=2000,
               T0_frac=0.05, T_final_frac=0.001,
               K_destroy=10,
               lr_sel=3e-4, print_every=50):
    """
    NDOR: Neural Destroy + OR-Tools Repair + C++ SA.

    Each episode:
      1. SelectionPolicy outputs destroy weights (per demand)
      2. C++ SA runs sa_iters_per_ep steps with those weights
         (SA automatically uses weights to bias which demands to touch)
      3. OR-Tools repairs K most-touched demands optimally
      4. REINFORCE update: reward = score improvement
    """
    opt_sel = torch.optim.Adam(sel_policy.parameters(), lr=lr_sel)

    # Build greedy initial route indices
    init_routings = init_sol.routings
    cpp_solver, cand_list = build_cpp_solver(env, dd, init_routings)

    best_score  = cpp_solver.get_score()
    best_routes = list(cpp_solver.get_routes())
    T0          = best_score * T0_frac
    T_final     = best_score * T_final_frac

    log = []
    print(f'\n── NDOR ──  init={best_score:,.0f}  episodes={n_episodes}')
    print(f'  SA: {sa_iters_per_ep} iters/ep  OR-Tools: K={K_destroy} repair')

    for ep in range(1, n_episodes+1):
        frac   = ep / n_episodes
        T_ep   = T0 * ((T_final/T0) ** frac)
        rl_tmp = max(0.1, 1.0 - frac*0.9)

        # ── Step 1: SelectionPolicy → destroy weights ──────────────────────────
        sel_policy.eval()
        route_idx = list(cpp_solver.get_routes())
        sel_feats, valid_idx = get_sel_feats(dd, route_idx)

        feat_t   = torch.tensor(sel_feats, dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            scores = sel_policy(feat_t)   # (N_valid,)
            # Convert to per-demand weights (full array for C++)
        weights_full = [1e-6] * len(dd)
        scores_np    = torch.softmax(scores / rl_tmp, dim=0).cpu().numpy()
        for j, i in enumerate(valid_idx):
            weights_full[i] = float(scores_np[j])

        # ── Step 2: C++ SA with RL-guided destroy weights ──────────────────────
        score_before = cpp_solver.get_score()
        sa_result = cpp_solver.sa_run(
            weights_full, T_ep, T_ep*0.01,
            sa_iters_per_ep, sa_iters_per_ep//10,
            seed=ep
        )
        score_after_sa = sa_result.best_score

        # Apply best SA solution
        cpp_solver.set_routes(sa_result.best_routes)

        # ── Step 3: OR-Tools repair on K most-active demands ──────────────────
        # Find K demands most frequently selected by SA
        from collections import Counter
        demand_counts = Counter(sa_result.selected_demands)
        top_k = [i for i, _ in demand_counts.most_common(K_destroy)
                 if dd[i] is not None]

        if top_k:
            current_routings = route_idx_to_routing(list(cpp_solver.get_routes()), dd, env)
            repaired = ortools_repair(top_k, dd, env, current_routings)
            # Re-init solver with repaired solution
            cpp_solver2, _ = build_cpp_solver(env, dd, repaired)
            if cpp_solver2.get_score() < cpp_solver.get_score():
                cpp_solver = cpp_solver2

        score_after = cpp_solver.get_score()

        if score_after < best_score:
            best_score  = score_after
            best_routes = list(cpp_solver.get_routes())

        # ── Step 4: REINFORCE update ──────────────────────────────────────────
        reward = (score_before - score_after) / max(score_before, 1.0)

        sel_policy.train()
        feat_t2  = torch.tensor(sel_feats, dtype=torch.float32, device=DEVICE)
        scores2  = sel_policy(feat_t2)
        dist     = Categorical(logits=scores2 / rl_tmp)
        # Use the top_k selections as "actions" for REINFORCE
        sel_loss = torch.tensor(0.0, device=DEVICE)
        for i in top_k:
            if i in valid_idx:
                j = valid_idx.index(i)
                lp = dist.log_prob(torch.tensor(j, device=DEVICE))
                sel_loss = sel_loss + (-lp * reward - 0.01 * dist.entropy())

        opt_sel.zero_grad()
        if sel_loss.requires_grad:
            sel_loss.backward()
            torch.nn.utils.clip_grad_norm_(sel_policy.parameters(), 1.0)
            opt_sel.step()

        log.append({'ep': ep, 'score': score_after, 'best': best_score,
                    'reward': reward, 'sa_improve': score_before - score_after_sa})

        if ep % print_every == 0:
            print(f'  ep {ep:4d} | score={score_after:>16,.0f} | '
                  f'best={best_score:>16,.0f} | rew={reward:+.4f}')

    # Return best solution
    cpp_solver.set_routes(best_routes)
    best_routings = route_idx_to_routing(best_routes, dd, env)
    print(f'  Final best: {best_score:,.0f}')
    return best_routings, best_score, log


print('NDOR training loop ready.')

## 6. Train: L1 → L2 → L3

In [ ]:
CASE_ORDER = [
    ('l1', 0.5), ('l1', 1.0), ('l1', 2.0),
    ('l2', 0.5), ('l2', 1.0), ('l2', 2.0),
    ('l3', 0.5), ('l3', 1.0), ('l3', 2.0),
]

NDOR_EPISODES  = {'l1': 300, 'l2': 150, 'l3': 80}
SA_ITERS_PER_EP= {'l1': 5000, 'l2': 10000, 'l3': 20000}
K_DESTROY      = {'l1': 5,  'l2': 20, 'l3': 50}
BC_EPOCHS      = {'l1': 200, 'l2': 50, 'l3': 30}

solutions  = {}
train_logs = {}

import os
os.makedirs('saved_models', exist_ok=True)

In [ ]:
# ── L1 ────────────────────────────────────────────────────────────────────────

sel_l1 = None  # carry over for transfer

for mult in [0.5, 1.0, 2.0]:
    tag = f'l1_{mult}'
    print(f'\n{"="*60}\n  L1 × {mult}\n{"="*60}')
    t0 = time.time()

    env = load_env('l1', mult)
    dd  = precompute(env)

    sel_policy, rout_policy = make_policies()
    if sel_l1 is not None:
        sel_policy.load_state_dict(sel_l1)  # warm-start selection from prev mult

    init_sol, g0 = pretrain_bc(rout_policy, env, dd, n_epochs=BC_EPOCHS['l1'])
    init_score, _ = evaluate(init_sol, env['net'], env['od_matrix'],
                             env['settings'], env['yard_info'])

    best_routings, best_score, log = ndor_train(
        sel_policy, env, dd, init_sol, init_score,
        n_episodes=NDOR_EPISODES['l1'],
        sa_iters_per_ep=SA_ITERS_PER_EP['l1'],
        K_destroy=K_DESTROY['l1'],
    )
    train_logs[tag] = log
    sel_l1 = deepcopy(sel_policy.state_dict())

    sol = Solution(env['demands'], best_routings)
    score, stats = evaluate(sol, env['net'], env['od_matrix'],
                            env['settings'], env['yard_info'])
    print(f'  stress={stats["stress_score"]:,.0f}  blocks={stats["n_blocks"]}')

    solutions[tag] = build_json(sol, env['net'], env['od_matrix'], env['settings'],
                                env['nodes_df'], env['links_df'], env['demands_raw'])
    torch.save(sel_policy.state_dict(), f'saved_models/ndor_sel_l1_{mult}.pt')
    print(f'  Elapsed: {time.time()-t0:.1f}s')

In [ ]:
# ── L2 (transfer from L1) ──────────────────────────────────────────────────────

for mult in [0.5, 1.0, 2.0]:
    tag = f'l2_{mult}'
    print(f'\n{"="*60}\n  L2 × {mult}\n{"="*60}')
    t0 = time.time()

    env = load_env('l2', mult)
    dd  = precompute(env)

    sel_policy, rout_policy = make_policies()
    sel_policy.load_state_dict(torch.load(f'saved_models/ndor_sel_l1_1.0.pt', map_location=DEVICE))
    print('  Transfer: L1 selection weights')

    init_sol, g0 = pretrain_bc(rout_policy, env, dd, n_epochs=BC_EPOCHS['l2'])
    init_score, _ = evaluate(init_sol, env['net'], env['od_matrix'],
                             env['settings'], env['yard_info'])

    best_routings, best_score, log = ndor_train(
        sel_policy, env, dd, init_sol, init_score,
        n_episodes=NDOR_EPISODES['l2'],
        sa_iters_per_ep=SA_ITERS_PER_EP['l2'],
        K_destroy=K_DESTROY['l2'],
        lr_sel=1e-4,
    )
    train_logs[tag] = log

    sol = Solution(env['demands'], best_routings)
    score, stats = evaluate(sol, env['net'], env['od_matrix'],
                            env['settings'], env['yard_info'])
    print(f'  stress={stats["stress_score"]:,.0f}  blocks={stats["n_blocks"]}')

    solutions[tag] = build_json(sol, env['net'], env['od_matrix'], env['settings'],
                                env['nodes_df'], env['links_df'], env['demands_raw'])
    torch.save(sel_policy.state_dict(), f'saved_models/ndor_sel_l2_{mult}.pt')
    print(f'  Elapsed: {time.time()-t0:.1f}s')

In [ ]:
# ── L3 (transfer from L2) ──────────────────────────────────────────────────────

for mult in [0.5, 1.0, 2.0]:
    tag = f'l3_{mult}'
    print(f'\n{"="*60}\n  L3 × {mult}\n{"="*60}')
    t0 = time.time()

    env = load_env('l3', mult)
    dd  = precompute(env)

    sel_policy, rout_policy = make_policies()
    sel_policy.load_state_dict(torch.load(f'saved_models/ndor_sel_l2_1.0.pt', map_location=DEVICE))
    print('  Transfer: L2 selection weights')

    init_sol, g0 = pretrain_bc(rout_policy, env, dd, n_epochs=BC_EPOCHS['l3'])
    init_score, _ = evaluate(init_sol, env['net'], env['od_matrix'],
                             env['settings'], env['yard_info'])

    best_routings, best_score, log = ndor_train(
        sel_policy, env, dd, init_sol, init_score,
        n_episodes=NDOR_EPISODES['l3'],
        sa_iters_per_ep=SA_ITERS_PER_EP['l3'],
        K_destroy=K_DESTROY['l3'],
        lr_sel=3e-5,
    )
    train_logs[tag] = log

    sol = Solution(env['demands'], best_routings)
    score, stats = evaluate(sol, env['net'], env['od_matrix'],
                            env['settings'], env['yard_info'])
    print(f'  stress={stats["stress_score"]:,.0f}  blocks={stats["n_blocks"]}')

    solutions[tag] = build_json(sol, env['net'], env['od_matrix'], env['settings'],
                                env['nodes_df'], env['links_df'], env['demands_raw'])
    torch.save(sel_policy.state_dict(), f'saved_models/ndor_sel_l3_{mult}.pt')
    print(f'  Elapsed: {time.time()-t0:.1f}s')

## 7. Generate submission.csv

In [ ]:
rows = [['ID', 'data']]
for case_id, (layer, mult) in enumerate(CASE_ORDER):
    tag = f'{layer}_{mult}'
    sol = solutions.get(tag)
    if sol is None:
        print(f'[ID {case_id}] {tag} MISSING')
        rows.append([case_id, '{}']); continue
    data_str = json.dumps(sol, separators=(',', ':'))
    print(f'[ID {case_id}] {tag}  blocks={len(sol["outputs"]["1 Block Design"])}  len={len(data_str):,}')
    rows.append([case_id, data_str])

with open('submission.csv', 'w', newline='', encoding='utf-8') as f:
    csv.writer(f).writerows(rows)
print(f'Done. {sum(len(r[1]) for r in rows[1:])/1e6:.1f} MB')

from google.colab import files
files.download('submission.csv')

## 8. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (layer, mult) in zip(axes, [('l1',1.0),('l2',1.0),('l3',1.0)]):
    log = train_logs.get(f'{layer}_{mult}', [])
    if not log: continue
    ax.plot([d['ep'] for d in log], [d['best'] for d in log])
    ax.set_title(f'{layer.upper()} ×{mult}')
    ax.set_xlabel('Episode'); ax.set_ylabel('Best Stress Score')
    ax.grid(True, alpha=0.3)
plt.suptitle('NDOR Training Curves (Neural Destroy + OR-Tools Repair + C++ SA)')
plt.tight_layout()
plt.savefig('training_curves_ndor.png', dpi=150)
plt.show()